# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [81]:
import os
import sys
import subprocess
import pandas as pd

# -----------------------------
# Setup
# -----------------------------
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

assert os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
), "Starter CSV not found."
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking

This lane is best framed as a ranking problem because the goal is to prioritize content pages for review rather than simply classify them as needing review or not. The model should estimate which pages deserve attention first based on search performance and engagement signals, allowing the SEO team to review the highest-priority pages before lower-priority ones.

In [82]:
print("Trend direction distribution:")
display(df["trend_direction"].value_counts())

print("\nAverage search position:")
display(df["avg_position"].describe())


Trend direction distribution:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Average search position:


,avg_position
count,30000.00000
mean,16.34238
std,15.21679
min,0.00000
25%,6.20000
50%,10.80000
75%,22.30000
max,245.00000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target (Proxy):

The proxy target is a review priority score that estimates how urgently a content page should be reviewed. Since the true business value of refreshing a page cannot be directly observed beforehand, this score serves as a measurable proxy derived from observed search performance and engagement signals.

In [83]:
proxy_columns = [
    "avg_position",
    "ctr",
    "trend_direction"
]

display(df[proxy_columns].head())

,avg_position,ctr,trend_direction
0,10.6,0.76,down
1,20.3,0.05,down
2,36.5,0.09,down
3,6.2,0.49,stable
4,44.0,0.13,down


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric:

Precision@20 is an appropriate metric because the SEO team is likely to review only a limited number of pages at a time. A high Precision@20 means that most of the top-ranked pages genuinely deserve review, making the prioritization effective.

In [84]:
print("Total content pages:", len(df))

print("\nTop pages by worst average position:")
display(
    df.sort_values("avg_position", ascending=False)[
        ["content_id", "avg_position", "ctr"]
    ].head(20)
)

Total content pages: 30000

Top pages by worst average position:


,content_id,avg_position,ctr
24445,content_661e1745db72,245.0,0.0
19920,content_23f1cc8851a9,184.0,0.0
26873,content_7275a6a3a8eb,165.5,0.0
16044,content_71a31b831092,161.0,0.0
18532,content_42c7c72b8391,145.5,0.0
15639,content_cb6c7d58c0bc,144.5,0.0
27923,content_692fda8c52bd,142.0,0.0
2970,content_3e087a5d8f15,138.8,0.0
28214,content_13bbd72aea33,118.0,0.0
1534,content_abeb1aa40158,113.5,0.0


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Each row in the dataset represents one content page together with its observed search performance and engagement metrics. These measurements are used as features to estimate the page's review priority.

In [85]:
display(df.head())
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Rows: 30000
Columns: 44


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule cannot capture the complex relationships between multiple search and engagement signals. A page with a poor average position may still perform well because of strong engagement or an improving trend. Machine learning can learn these interactions from data and produce a more accurate ranking than a single manually defined rule.

In [86]:
print(
    df[
        [
            "avg_position",
            "ctr",
            "trend_direction"
        ]
    ].corr(numeric_only=True)
)

              avg_position      ctr
avg_position       1.00000 -0.07259
ctr               -0.07259  1.00000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.